# Aegis-Finance: Phase 1 Data Audit
**Objective:** Assess reliability and integrity of the raw training dataset on Vertex AI.

In [ ]:
import pandas as pd
import os
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns

# Set aesthetic parameters for premium display
sns.set_theme(style="darkgrid", palette="pastel")

# 1. Locate and Load File Data
data_path = os.path.expanduser('~/Aegis-Finance/data/raw/train_data.parquet')
print(f"Scanning path: {data_path}\n")

try:
    parquet_file = pq.ParquetFile(data_path)
    df = next(parquet_file.iter_batches(batch_size=100000)).to_pandas()

    # 2. Executive Integrity Scan Summary Table
    missing_pct = (df.isnull().sum() / len(df)) * 100
    
    summary_df = pd.DataFrame({
        'Data Type': df.dtypes,
        'Missing %': missing_pct,
        'Number of Unique Values': df.nunique()
    }).reset_index().rename(columns={'index': 'Column Name'})
    
    print("--- INTEGRITY SCAN SUMMARY ---")
    display(summary_df)
    
    # 3. Target Variable CountPlot
    if 'target' in df.columns:
        plt.figure(figsize=(8, 5))
        ax = sns.countplot(data=df, x='target', edgecolor='black')
        plt.title('Distribution of Defaults (Target)', fontsize=14, fontweight='bold')
        plt.xlabel('Target (0 = No Default, 1 = Default)', fontsize=12)
        plt.ylabel('Count', fontsize=12)
        
        # Add percentage labels
        total = len(df)
        for p in ax.patches:
            percentage = f'{100 * p.get_height() / total:.1f}%'
            x = p.get_x() + p.get_width() / 2 - 0.05
            y = p.get_y() + p.get_height() + (total * 0.01)
            ax.annotate(percentage, (x, y), ha='center')
            
        plt.show()
    else:
        print("\n[!] MLOps WARN: 'target' column missing. Unable to plot.")

except FileNotFoundError:
    print("\n[!] FileNotFoundError: System could not locate the Parquet file.")
except StopIteration:
    print("\n[!] The dataset is entirely empty.")
except Exception as e:
    print(f"\n[!] Critical Runtime Error: {e}")


### 🛡️ Senior Council Analysis: Handling Delinquency (D_) Variables

**Observation:** Many of the `D_*` (Delinquency) columns will exhibit substantial sparsity (> 50% empty).

**Architect & MLOps Recommendation:**
Do **NOT** use Simple Imputation (Mean/Median) for these features. Given the sparsity, mean/median will aggressively distort the true statistical distribution and dilute tree-split purity.

**Target Engineering Protocol:**
1. **Create a Binary Mask:** For every sparse feature, engineer an `is_missing` flag (e.g. `D_42_is_missing` = 1/0).
2. **Sentinel/Native NaN:** Replace the original NaNs with an extreme sentinel value (e.g., `-999`) or natively leave them as `NaN` relying on the internal missing-value handlers of LightGBM / XGBoost.
